# Türkçe GPT — Colab'da sıfırdan eğitim

Bu defter, [turkce-gptr](https://github.com/mustafagulturk/turkce-gptr) projesindeki küçük Türkçe dil modelini Google Colab'ın **ücretsiz T4 GPU**'sunda baştan sona eğitir:

1. GPU kontrolü
2. Projeyi indirme ve kurulum
3. Türkçe Wikipedia'dan veri hazırlama (temizlik + BPE tokenizer eğitimi)
4. `kucuk` boyutta modeli ön eğitme (~1 saat)
5. Loss grafiği ve değerlendirme
6. Metin üretimi
7. Soru-cevap verisiyle ince ayar (sohbet modeli)
8. Sohbet
9. Modeli Google Drive'a kaydetme

> **Başlamadan önce:** Menüden *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* seçin. Sonra hücreleri sırayla çalıştırın.

## 1. GPU kontrolü

`nvidia-smi` ekran kartını ve belleğini gösterir. Ardından PyTorch'un GPU'yu görüp görmediğini kontrol ediyoruz. `False` görürseniz çalışma zamanı türünü GPU yapmayı unutmuşsunuzdur; eğitim işlemcide saatler sürer.

In [ ]:
!nvidia-smi

import torch
print("PyTorch sürümü:", torch.__version__)
print("CUDA kullanılabilir mi?", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Projeyi indirme

Kodu GitHub'dan klonlayıp proje klasörüne geçiyoruz. Bütün komutlar `python -m turkce_gpt <komut>` biçiminde çalışır; her komutun seçeneklerini `--help` ile görebilirsiniz.

In [ ]:
!git clone https://github.com/mustafagulturk/turkce-gptr.git
%cd /content/turkce-gptr
!python -m turkce_gpt

## 3. Kurulum

Colab'da PyTorch ve NumPy zaten kurulu. Wikipedia verisini HuggingFace'ten indirmek için sadece `datasets` kütüphanesi gerekiyor.

In [ ]:
!pip install -q datasets

## 4. Veri hazırlama: Türkçe Wikipedia

Bu adım:
- Türkçe Wikipedia makalelerini indirir ve temizler (en fazla **300 MB** metin),
- metinden **16384** tokenlik bir BPE sözlüğü öğrenir (`--sozluk`),
- tüm metni token numaralarına çevirip `veri/hazir/` klasörüne `train.bin` ve `val.bin` olarak yazar.

Belgelerin arasına `<|son|>` tokeni konur; model böylece bir metnin nerede bittiğini öğrenir. Bu adım internet hızına göre birkaç dakika ile yarım saat arasında sürebilir.

In [ ]:
!python -m turkce_gpt veri --kaynak wikipedia --max-mb 300 --cikti veri/hazir --sozluk 16384
!cat veri/hazir/meta.json

## 5. Ön eğitim (`kucuk` boyut)

`kucuk` boyut: 8 katman, 8 dikkat başlığı, 512 genişlik, 512 tokenlik bağlam (sözlükle birlikte ~30M parametre).

Seçenekler:
- `--batch 16 --birikim 2`: her adımda 2 × 16 = 32 dizi (32 × 512 ≈ 16 bin token) görülür. Gradyan biriktirme, T4'ün 16 GB belleğine sığmayan büyük bir yığını iki parçada işlememizi sağlar.
- `--adim 5000`: T4'te yaklaşık 1 saat. İlk birkaç yüz adımdaki hızı (adım başına süre) izleyin; çok yavaşsa `--adim`'i azaltın.
- `--lr 6e-4 --isinma 200`: öğrenme oranı ilk 200 adımda sıfırdan yavaşça yükselir (ısınma), sonra kosinüs eğrisiyle azalır.
- `--degerlendirme-araligi 250`: her 250 adımda doğrulama kaybı (val loss) ölçülür, en iyi model `modeller/kucuk/model.pt`'ye kaydedilir ve `log.csv`'ye bir satır yazılır.

T4 bfloat16 desteklemediği için eğitim kodu otomatik olarak float16 + GradScaler kullanır. Bağlantı koparsa ve dosyalar duruyorsa aynı komutu `--devam` ekleyerek çalıştırıp kaldığı yerden sürdürebilirsiniz.

In [ ]:
!python -m turkce_gpt egit --veri veri/hazir --boyut kucuk --cikti modeller/kucuk \
    --adim 5000 --batch 16 --birikim 2 --lr 6e-4 --isinma 200 \
    --degerlendirme-araligi 250 --cihaz auto

## 6. Loss grafiği

Eğitim `modeller/kucuk/log.csv` dosyasına her değerlendirmede `adim, train_loss, val_loss, lr, sure` sütunlarını yazar. İki eğrinin birlikte düşmesi modelin öğrendiğini gösterir. Val loss düşmeyi bırakıp train loss düşmeye devam ederse model ezberlemeye (overfitting) başlamış demektir.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv("modeller/kucuk/log.csv")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(log["adim"], log["train_loss"], label="eğitim (train)")
ax1.plot(log["adim"], log["val_loss"], label="doğrulama (val)")
ax1.set_xlabel("adım")
ax1.set_ylabel("loss")
ax1.set_title("Kayıp (loss)")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(log["adim"], log["lr"])
ax2.set_xlabel("adım")
ax2.set_title("Öğrenme oranı (ısınma + kosinüs)")
ax2.grid(alpha=0.3)
plt.show()

print(log.tail())

### Değerlendirme

Doğrulama verisinde kayıp (loss), perplexity ve bits-per-byte ölçülür. Perplexity kabaca "model her adımda kaç seçenek arasında kararsız kalıyor" demektir; ne kadar düşükse o kadar iyi.

In [ ]:
!python -m turkce_gpt degerlendir --model modeller/kucuk/model.pt --veri veri/hazir

## 7. Metin üretimi

Model verilen başlangıcı devam ettirir. Tokenler üretildikçe ekrana basılır.
- `--sicaklik`: düşükse (0.5) temkinli ve tekrarcı, yüksekse (1.0+) yaratıcı ama dağınık.
- `--top-k` / `--top-p`: sadece en olası tokenler arasından seçerek saçma kelimeleri eler.
- `--tohum`: aynı tohum aynı metni üretir; farklı sonuç için değiştirin.

In [ ]:
!python -m turkce_gpt uret --model modeller/kucuk/model.pt --baslangic "Türkiye'nin en büyük şehirlerinden biri olan" --uzunluk 200 --sicaklik 0.8 --top-k 50 --top-p 0.95 --tohum 42

In [ ]:
!python -m turkce_gpt uret --model modeller/kucuk/model.pt --baslangic "Gaziantep" --uzunluk 150 --sicaklik 0.7 --top-k 50 --top-p 0.95 --tohum 7

## 8. İnce ayar: sohbet modeli

Ön eğitimli model sadece "metni devam ettirmeyi" bilir. Soru sorulduğunda cevap vermesi için onu `veri/sohbet.jsonl`'deki elle yazılmış ~180 Türkçe soru-cevap örneğiyle ince ayar yapıyoruz. Şablon:

```
<|kullanici|>soru<|asistan|>cevap<|son|>
```

Eğitimde sadece **cevap** kısmının kaybı hesaplanır; model kullanıcı mesajını tahmin etmeyi değil, cevap vermeyi öğrenir. Öğrenme oranı ön eğitimden çok daha küçüktür (`1e-4`), böylece model önceden öğrendiği Türkçeyi unutmaz.

In [ ]:
!python -m turkce_gpt ince-ayar --model modeller/kucuk/model.pt --veri veri/sohbet.jsonl --cikti modeller/kucuk-sohbet --adim 300 --lr 1e-4

## 9. Sohbet

`sohbet` komutu normalde etkileşimlidir (`/temizle` geçmişi siler, `/cikis` çıkar). Burada mesajları `printf` ile sırayla göndererek örnek bir konuşma yapıyoruz. Etkileşimli denemek isterseniz `!python -m turkce_gpt sohbet --model modeller/kucuk-sohbet/model.pt` komutunu çalıştırıp çıkan kutuya yazabilirsiniz.

Not: Küçük bir model olduğu için cevaplar her zaman doğru olmayacaktır; özellikle ince ayar verisinde olmayan sorularda uydurabilir.

In [ ]:
!printf "Merhaba!\nSen kimsin?\nGaziantep neyle ünlüdür?\nTürkiye'nin başkenti neresidir?\n/cikis\n" | python -m turkce_gpt sohbet --model modeller/kucuk-sohbet/model.pt

### Python içinden sohbet

Aynı işi Python'dan da yapabiliriz. Bu hücre, komut satırı aracının içinde ne olduğunu gösterir: model dosyasını yükle, konuşmayı şablona göre tokenlere çevir, `generate_stream` ile tokenleri tek tek üret, yarım UTF-8 karakter basmamak için artımlı çöz.

In [ ]:
import torch
from turkce_gpt.checkpoint import load_checkpoint
from turkce_gpt.chat import encode_prompt
from turkce_gpt.interactive import IncrementalDecoder
from turkce_gpt.tokenizer import END_OF_TEXT

device = "cuda" if torch.cuda.is_available() else "cpu"
model, tokenizer, info = load_checkpoint("modeller/kucuk-sohbet/model.pt", device=device)

def sor(soru: str) -> None:
    ids = encode_prompt(tokenizer, [{"role": "kullanici", "content": soru}])
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    decoder = IncrementalDecoder(tokenizer)
    print("Soru:", soru)
    print("Cevap: ", end="")
    for token_id in model.generate_stream(idx, max_new_tokens=100, temperature=0.7, top_k=50, top_p=0.9,
                                          stop_ids={tokenizer.special_id(END_OF_TEXT)}):
        print(decoder.add(token_id), end="", flush=True)
    print(decoder.flush(), "\n")

sor("Fotosentez nedir?")
sor("Antep fıstığı nedir?")

## 10. Modeli Google Drive'a kaydetme

Colab oturumu kapanınca `/content` altındaki her şey silinir. Eğitilmiş modelleri Drive'a kopyalayalım. Her `model.pt` tek başına taşınabilir: ağırlıkları, model ayarlarını ve tokenizer'ı birlikte içerir. Daha sonra başka bir bilgisayarda `--model` ile doğrudan kullanabilirsiniz.

İsterseniz `veri/hazir` klasörünü de kopyalayıp bir sonraki oturumda veri hazırlama adımını atlayabilirsiniz.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/turkce-gpt
!cp -r modeller /content/drive/MyDrive/turkce-gpt/
!ls -lh /content/drive/MyDrive/turkce-gpt/modeller/*